# Swell direction from the 3C seismometers (Welch horizontal PCA)

Memory-safe chunked processing of 3C geophone data using Welch-style
frequency-domain horizontal PCA.

Input: the 10 Hz GP1/GP2/GPZ counts written by `main_decimate.ipynb`
(`X1.N?.10Hz.mseed`) and the station's StationXML (`X1.N?.xml`), both in
`data/seismometer_decimated/`.

Design:
- Process 2-day chunks
- Move forward by 1 day (50% overlap)
- Keep only the middle day
- Preprocess in time domain once per chunk
- For each 30-min block:
    - split into 10 snapshots of 180 s (no overlap)
    - compute Welch-averaged cross-spectral matrix
    - estimate azimuth/rectilinearity at each frequency bin
- Save one CSV row per (time block, frequency)

Notes:
- This is the frequency-domain analogue of horizontal PCA.
- It is not the exact time-domain Flinn algorithm.
- Azimuth is estimated from the dominant eigenvector of the REAL PART
  of the horizontal 2x2 spectral matrix.
- Power is saved as total 3C PSD: Pzz + Pnn + Pee.

In [1]:
import gc
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
from obspy import UTCDateTime, read, read_inventory

from das_tools.utils import proj_path

## Parameters

In [ ]:
seism_data_dir = proj_path.data_dir("seismometer_decimated")   # holds X1.N<sensor_no>.10Hz.mseed and X1.N<sensor_no>.xml
sensor_no = 5
t_start = datetime.fromisoformat("2025-04-24")
t_end = datetime.fromisoformat("2025-05-08")
OUT = proj_path.output_dir("seismometer")

# preprocessing
target_fs = 5.0
aa_cutoff = 1.8
pre_filt = (0.02, 0.04, 1.2, 2.0)
water_level = 60.0

# Welch blocks: nsnap x snapshot_sec = 30 min
snapshot_sec = 180.0
nsnap = 10

freq_min = 0.05
freq_max = 1.0

## Preprocessing

In [3]:
def preprocess_stream(st, inv, target_fs, aa_cutoff, pre_filt, water_level):
    st = st.copy()
    st.merge(method=1, fill_value="interpolate")
    st.detrend("demean")
    st.detrend("linear")
    st.taper(max_percentage=0.02)

    st.filter("lowpass", freq=aa_cutoff, corners=4, zerophase=True)
    # integer decimation, the lowpass above is the anti-alias filter. Not
    # st.resample: from 10 Hz input its FFT window and spectrum interpolation
    # take 20-30% of the power at 1 Hz
    factor = st[0].stats.sampling_rate / target_fs
    if abs(factor - round(factor)) > 1e-9:
        raise ValueError(f"sampling rate {st[0].stats.sampling_rate} Hz is not a multiple of target_fs")
    st.decimate(int(round(factor)), no_filter=True)

    st.remove_response(
        inventory=inv,
        output="VEL",
        pre_filt=tuple(pre_filt),
        water_level=water_level,
    )

    st.rotate(method="->ZNE", inventory=inv)

    for tr in st:
        tr.data = tr.data.astype(np.float32)

    return st

## Welch helpers

In [4]:
def one_sided_scale(spec, nfft):
    """
    Convert two-sided FFT power/cross-power to one-sided.
    """
    spec = spec.copy()
    if nfft % 2 == 0:
        # even nfft: keep DC and Nyquist unchanged
        if len(spec) > 2:
            spec[1:-1] *= 2.0
    else:
        # odd nfft: keep DC unchanged
        if len(spec) > 1:
            spec[1:] *= 2.0
    return spec


def welch_cross_spectra(x_block, y_block, fs, snapshot_n, nsnap):
    """
    Welch-average cross-spectrum from a block containing nsnap snapshots,
    each of length snapshot_n samples, no overlap.

    Returns
    -------
    f : ndarray
        Frequency vector (Hz)
    Pxy : ndarray of complex
        One-sided averaged cross-spectrum
    """
    x_seg = x_block[: nsnap * snapshot_n].reshape(nsnap, snapshot_n)
    y_seg = y_block[: nsnap * snapshot_n].reshape(nsnap, snapshot_n)

    win = np.hanning(snapshot_n).astype(np.float64)
    U = np.sum(win ** 2)

    Pxy_acc = None

    for k in range(nsnap):
        xk = x_seg[k].astype(np.float64)
        yk = y_seg[k].astype(np.float64)

        xk = xk - np.mean(xk)
        yk = yk - np.mean(yk)

        Xk = np.fft.rfft(xk * win)
        Yk = np.fft.rfft(yk * win)

        Pxy_k = (Xk * np.conj(Yk)) / (fs * U)
        Pxy_k = one_sided_scale(Pxy_k, snapshot_n)

        if Pxy_acc is None:
            Pxy_acc = Pxy_k
        else:
            Pxy_acc += Pxy_k

    Pxy = Pxy_acc / nsnap
    f = np.fft.rfftfreq(snapshot_n, d=1.0 / fs)
    return f, Pxy

## Frequency-domain horizontal PCA

In [5]:
def analyze_chunk_welch(
    st_zne,
    snapshot_sec=180,
    nsnap=10,
    freq_min=0.0,
    freq_max=None,
):
    """
    Frequency-domain horizontal PCA using Welch-averaged spectral matrix.

    For each 30-min block (= nsnap * snapshot_sec):
      - compute Welch-averaged spectra/cross-spectra
      - build 2x2 horizontal spectral matrix at each frequency
      - estimate azimuth and rectilinearity
      - save 3C power spectrum

    Returns
    -------
    DataFrame with columns:
      t_mid, freq_hz, azimuth_deg_180, rectilinearity, power_3c, power_h
    """
    trZ = st_zne.select(component="Z")[0]
    trN = st_zne.select(component="N")[0]
    trE = st_zne.select(component="E")[0]

    fs = trZ.stats.sampling_rate
    if not (trN.stats.sampling_rate == fs and trE.stats.sampling_rate == fs):
        raise ValueError("Z, N, E must have the same sampling rate")

    z = trZ.data
    n = trN.data
    e = trE.data
    t0 = trZ.stats.starttime

    snapshot_n = int(snapshot_sec * fs)
    block_n = nsnap * snapshot_n
    step_n = block_n  # every 30 min, since 10 x 180 s snapshots

    rows = []

    for i0 in range(0, len(z) - block_n + 1, step_n):
        i1 = i0 + block_n

        zb = z[i0:i1]
        nb = n[i0:i1]
        eb = e[i0:i1]

        f, Pee = welch_cross_spectra(eb, eb, fs, snapshot_n, nsnap)
        _, Pnn = welch_cross_spectra(nb, nb, fs, snapshot_n, nsnap)
        _, Pzz = welch_cross_spectra(zb, zb, fs, snapshot_n, nsnap)
        _, Pen = welch_cross_spectra(eb, nb, fs, snapshot_n, nsnap)

        t_mid = t0 + (i0 + 0.5 * block_n) / fs

        for j, fj in enumerate(f):
            if fj <= 0.0:
                continue
            if fj < freq_min:
                continue
            if freq_max is not None and fj > freq_max:
                continue

            # Horizontal spectral matrix, using real part for line orientation
            See = float(np.real(Pee[j]))
            Snn = float(np.real(Pnn[j]))
            Sen = float(np.real(Pen[j]))

            H = np.array([
                [See, Sen],
                [Sen, Snn],
            ], dtype=np.float64)

            eigvals, eigvecs = np.linalg.eigh(H)
            lam2, lam1 = eigvals
            v = eigvecs[:, 1]  # dominant eigenvector in [E, N]

            azimuth = np.degrees(np.arctan2(v[0], v[1])) % 180.0
            rect = (lam1 - lam2) / (lam1 + lam2 + 1e-20)

            power_h = See + Snn
            power_3c = See + Snn + float(np.real(Pzz[j]))

            rows.append({
                "t_mid": pd.Timestamp(str(t_mid.datetime)),
                "freq_hz": float(fj),
                "azimuth_deg_180": float(azimuth),
                "rectilinearity": float(rect),
                "power_h": float(power_h),
                "power_3c": float(power_3c),
            })

        del Pee, Pnn, Pzz, Pen

    return pd.DataFrame(rows)

## Inventory, orientation and decimated data

In [6]:
OUT.mkdir(parents=True, exist_ok=True)

inv = read_inventory(str(seism_data_dir / f"X1.N{sensor_no}.xml"))

# Orientation
for chan in inv[0][0]:
    if chan.code == "GP1":
        chan.azimuth = 0.0
    elif chan.code == "GP2":
        chan.azimuth = 90.0

# 10 Hz counts from main_decimate.ipynb; gaps are separate traces
st_all = read(str(seism_data_dir / f"X1.N{sensor_no}.10Hz.mseed"))
print(st_all)

4 Trace(s) in Stream:
X1.N4..GP1 | 2025-04-25T19:53:20.000000Z - 2025-04-27T16:56:15.800000Z | 10.0 Hz, 1621759 samples
X1.N4..GP1 | 2025-04-28T00:00:10.000000Z - 2025-05-07T21:19:15.900000Z | 10.0 Hz, 8543460 samples
X1.N4..GP2 | 2025-04-25T19:53:20.000000Z - 2025-05-07T21:19:15.900000Z | 10.0 Hz, 10419560 samples
X1.N4..GPZ | 2025-04-25T19:53:20.000000Z - 2025-05-07T21:19:15.900000Z | 10.0 Hz, 10419560 samples


## Run: 2-day chunks, 1-day step, keep the middle day

Rows are appended to the CSV chunk by chunk; an existing file for the sensor is replaced.

In [7]:
chunk_len = timedelta(days=2)
step = timedelta(days=1)

out_file = OUT / f"welch_pca_results_N{sensor_no}.csv"
out_file.unlink(missing_ok=True)  # chunks are appended below, so start each run fresh
wrote_header = False

cur = t_start
while cur < t_end:
    chunk_start = cur
    chunk_end = min(cur + chunk_len, t_end)

    print(f"Processing chunk: {chunk_start} -> {chunk_end}")

    st = st_all.slice(UTCDateTime(chunk_start), UTCDateTime(chunk_end)).copy()

    if len(st) == 0:
        cur += step
        continue

    st = preprocess_stream(
        st,
        inv,
        target_fs=target_fs,
        aa_cutoff=aa_cutoff,
        pre_filt=pre_filt,
        water_level=water_level,
    )

    df_chunk = analyze_chunk_welch(
        st,
        snapshot_sec=snapshot_sec,
        nsnap=nsnap,
        freq_min=freq_min,
        freq_max=freq_max,
    )

    # Keep only the middle day
    keep_start = chunk_start + timedelta(days=0.5)
    keep_end = chunk_start + timedelta(days=1.5)

    df_chunk = df_chunk[
        (df_chunk["t_mid"] >= pd.Timestamp(keep_start)) &
        (df_chunk["t_mid"] < pd.Timestamp(keep_end))
    ]

    if len(df_chunk) > 0:
        df_chunk.to_csv(
            out_file,
            mode="a",
            header=not wrote_header,
            index=False,
        )
        wrote_header = True
        print(f"Appended {len(df_chunk)} rows to {out_file}")

    del st, df_chunk
    gc.collect()

    cur += step

print(f"Saved: {out_file}")

Processing chunk: 2025-04-24 00:00:00 -> 2025-04-26 00:00:00
Processing chunk: 2025-04-25 00:00:00 -> 2025-04-27 00:00:00
Appended 5504 rows to /Users/junsuj/Desktop/git/das-landfast-ice-inversion2025/outputs/seismometer/welch_pca_results_N4.csv
Processing chunk: 2025-04-26 00:00:00 -> 2025-04-28 00:00:00
Appended 8256 rows to /Users/junsuj/Desktop/git/das-landfast-ice-inversion2025/outputs/seismometer/welch_pca_results_N4.csv
Processing chunk: 2025-04-27 00:00:00 -> 2025-04-29 00:00:00
Appended 8256 rows to /Users/junsuj/Desktop/git/das-landfast-ice-inversion2025/outputs/seismometer/welch_pca_results_N4.csv
Processing chunk: 2025-04-28 00:00:00 -> 2025-04-30 00:00:00
Appended 8256 rows to /Users/junsuj/Desktop/git/das-landfast-ice-inversion2025/outputs/seismometer/welch_pca_results_N4.csv
Processing chunk: 2025-04-29 00:00:00 -> 2025-05-01 00:00:00
Appended 8256 rows to /Users/junsuj/Desktop/git/das-landfast-ice-inversion2025/outputs/seismometer/welch_pca_results_N4.csv
Processing chu